### YOLO Inference

Detects pollinators directly from full images using a YOLO object detector with SAHI tiling.
No motion-detection step — catches insects that move too little or too fast for frame-differencing.

**Input** — a folder of camera images organised as `IMAGE_ROOT/{camera_id}/*.JPG` · `models/yolo_best.pt`

**Output** — `outputs/inference/yolo_results/{RUN_NAME}/yolo_results.csv` + `yolo_crops/` images

---

**How to run:**

1. *(Colab only)* Run **Cell 0** — mounts Drive and extracts the zip
2. Run **Cell 1** — auto-detects paths, no edits needed
3. Edit and run **Cell 2** — the only cell you need to change:
   - `IMAGE_ROOT` — which images to run on (default: `data/evaluation/images/`; change to any camera folder)
   - `YOLO_WEIGHTS` — path to the `.pt` file (default: `models/yolo_best.pt`)
   - `YOLO_CLASSES` — must match the class order the model was trained with
   - `RUN_NAME` is auto-generated from a timestamp — no need to change it
   - `YOLO_CONFIG` — adjust `conf_threshold`, `use_sahi`, `strip_height` if needed
4. Run **Cell 3** — loads the YOLO model
5. Run **Cell 4** — loads detection functions, no edits needed
6. Run **Cell 5** — lists camera folders found in `IMAGE_ROOT`
7. Run **Cell 6** — main inference loop (prints progress per camera folder)
8. Run **Cell 7** — summary of detections per class


##### Cell 0 — Colab setup  ← **Colab only, skip if running locally**

Mounts Google Drive and extracts the project zip. Skip this cell when running locally.

**Before running on Colab:** upload `pollinator-colab.zip` to the root of your Google Drive.
The zip top-level folder must be named `pollinator-colab/`:
```
pollinator-colab/
  data/
    evaluation/images/{camera_name}/*.JPG   ← default IMAGE_ROOT (change in Cell 2)
    training/raw_images/{camera_name}/*.JPG ← or point IMAGE_ROOT here
  models/
    yolo_best.pt
```

Results are saved to Drive at:
`MyDrive/pollinator-colab/outputs/inference/yolo_results/{RUN_NAME}/`

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    import os, zipfile
    from pathlib import Path

    DRIVE_ROOT = Path('/content/drive/MyDrive')
    ZIP_PATH   = DRIVE_ROOT / 'pollinator-colab.zip'
    EXTRACT_TO = Path('/content/pollinator-colab')

    if not EXTRACT_TO.exists():
        print(f'Extracting {ZIP_PATH.name} to /content/ ...')
        with zipfile.ZipFile(ZIP_PATH) as z:
            z.extractall('/content/')
        print('✓ Extracted')
    else:
        print(f'✓ Already extracted: {EXTRACT_TO}')

    print('Installing dependencies...')
    os.system('pip install -q ultralytics sahi opencv-python-headless torch torchvision')
    print('✓ Dependencies ready')
else:
    print('Running locally — skip this cell.')


##### Cell 1 — Environment  *(no edits needed)*

**Local:** auto-detects the repo root via `git rev-parse --show-toplevel` — no path editing required.

**Colab:** uses the path extracted from the zip in Cell 0.

Sets `MODEL_DIR` and output root folders. `IMAGE_ROOT` (which images to process) is set in **Cell 2 — Config**.

In [ ]:
from pathlib import Path

if IN_COLAB:
    BASE_DIR   = EXTRACT_TO          # set in Cell 0
    DRIVE_BASE = DRIVE_ROOT / 'pollinator-colab'
else:
    import subprocess as _sp
    _git_root  = Path(_sp.check_output(
        ['git', 'rev-parse', '--show-toplevel'], text=True).strip())
    BASE_DIR   = _git_root / 'ml_pipelines' / 'notebooks' / 'pollinator_detection'
    DRIVE_BASE = BASE_DIR

MODEL_DIR         = BASE_DIR  / 'models'
LOCAL_RESULTS     = Path('/content/data') if IN_COLAB else BASE_DIR
YOLO_RESULTS_ROOT = LOCAL_RESULTS / 'outputs' / 'inference' / 'yolo_results'
YOLO_RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

print(f'Env        : {"Colab" if IN_COLAB else "Local"}')
print(f'BASE_DIR   : {BASE_DIR}  exists={BASE_DIR.exists()}')
print(f'MODEL_DIR  : {MODEL_DIR}  exists={MODEL_DIR.exists()}')


##### Cell 2 — Config  ← **edit this before every run**

**1. `IMAGE_ROOT`** — which images to process.
Default: `data/evaluation/images/`. Change to any folder that contains `{camera_id}/` subfolders with `.JPG` files.
Examples:
```python
IMAGE_ROOT = BASE_DIR / 'data' / 'evaluation' / 'images'   # evaluation set (default)
IMAGE_ROOT = BASE_DIR / 'data' / 'training' / 'raw_images' # raw training footage
IMAGE_ROOT = Path('/any/absolute/path')                     # any other folder
```

**2. `RUN_NAME`** — short label for this run. Change each time to avoid overwriting previous results.

**3. `YOLO_WEIGHTS`** — path to the YOLO `.pt` file. Default: `models/yolo_best.pt`.

**4. `YOLO_CLASSES`** — class names in the same order as the model was trained. Must match exactly or all detections will have wrong labels.

**5. `YOLO_CONFIG`** — detection parameters:
- `use_sahi` (True) — tiled inference, better for small insects in large images
- `conf_threshold` (0.2) — detections below this are discarded
- `strip_height` (120) — pixels cropped from bottom to remove camera OSD bar
- `save_crops` (True) — save cropped patches of each detection

In [ ]:
# ── Input images ─────────────────────────────────────────────────
# Default: evaluation set. Change to any folder of camera images.
# Expected layout: IMAGE_ROOT/**/{camera_folder}/*.JPG
# Any depth of nesting is supported — the deepest folders with JPGs are used.
# Example: raw_images/site_A/Vamy/p1/101/*.JPG  →  processed as one camera
IMAGE_ROOT = BASE_DIR / 'data' / 'evaluation' / 'images'
# IMAGE_ROOT = BASE_DIR / 'data' / 'raw_images'                # ← raw camera images
# IMAGE_ROOT = Path('/absolute/path/to/any/image/folder')       # ← example

# ── Run name — auto-generated timestamp, unique every run ──────
import datetime as _dt
_ts = _dt.datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_NAME = f'run_{_ts}'
# Option B — timestamp + label (uncomment to use):
# RUN_NAME = f'run_{_ts}_yolo_sahi'

YOLO_WEIGHTS   = MODEL_DIR / 'yolo_best.pt'
YOLO_CLASSES   = ['fly', 'butterfly', 'other']  # bumblebee excluded from YOLO training

YOLO_CONFIG = {
    'conf_threshold': 0.2,
    'nms_iou':        0.45,
    'strip_height':   120,    # px to crop from bottom (Wingscapes OSD bar)
    'use_sahi':       True,   # recommended for 3008x1692 images
    'sahi_slice':     640,
    'sahi_overlap':   0.2,
    'sahi_conf':      0.05,
    'save_crops':     True,
    'crop_pad_px':    20,
    'progress_every': 50,
}

RUN_DIR = YOLO_RESULTS_ROOT / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

assert YOLO_WEIGHTS.exists(), f'YOLO weights not found: {YOLO_WEIGHTS}'
print(f'RUN_NAME    : {RUN_NAME}')
print(f'IMAGE_ROOT  : {IMAGE_ROOT}  exists={IMAGE_ROOT.exists()}')
print(f'Results  →  : {RUN_DIR}')
print(f'Weights     : {YOLO_WEIGHTS.name}')
print(f'SAHI        : {YOLO_CONFIG["use_sahi"]}  '
      f'(slice={YOLO_CONFIG["sahi_slice"]}  overlap={YOLO_CONFIG["sahi_overlap"]})')
print(f'Conf thresh : {YOLO_CONFIG["conf_threshold"]}')


##### Cell 3 — Load model
Loads YOLO weights and initialises the SAHI tiler. Prints a warning if SAHI is not installed.

In [ ]:
from ultralytics import YOLO
import sys, csv, time, cv2, numpy as np, json
from pathlib import Path
from PIL import Image as _PIL

print('Loading YOLO model...')
yolo_model = YOLO(str(YOLO_WEIGHTS))
print(f'  ✓ YOLO loaded: {YOLO_WEIGHTS.name}')

sahi_model = None
if YOLO_CONFIG['use_sahi']:
    try:
        from sahi import AutoDetectionModel
        from sahi.predict import get_sliced_prediction
        sahi_model = AutoDetectionModel.from_pretrained(
            model_type='ultralytics',
            model_path=str(YOLO_WEIGHTS),
            confidence_threshold=YOLO_CONFIG['sahi_conf'],
            device='cuda' if __import__('torch').cuda.is_available() else 'cpu')
        print(f'  ✓ SAHI loaded — tiling {YOLO_CONFIG["sahi_slice"]}px '
              f'overlap={YOLO_CONFIG["sahi_overlap"]}')
    except ImportError:
        print('  ⚠ SAHI not installed — falling back to direct YOLO.')
        print('    pip install sahi   to enable tiling.')


##### Cell 4 — Detection functions
Defines `detect_yolo` and `run_yolo_folder`. No edits needed.

In [ ]:
YOLO_CSV = [
    'camera_folder', 'image_name', 'crop_filename',
    'bbox_x', 'bbox_y', 'bbox_w', 'bbox_h',
    'class_name', 'confidence', 'method',
]

def detect_yolo(img_bgr, cfg):
    if sahi_model:
        res = get_sliced_prediction(
            _PIL.fromarray(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)),
            sahi_model,
            slice_height=cfg['sahi_slice'], slice_width=cfg['sahi_slice'],
            overlap_height_ratio=cfg['sahi_overlap'],
            overlap_width_ratio=cfg['sahi_overlap'],
            postprocess_type='NMM',
            postprocess_match_threshold=cfg['nms_iou'],
            verbose=0)
        return [{'x':int(o.bbox.minx), 'y':int(o.bbox.miny),
                 'w':int(o.bbox.maxx-o.bbox.minx),
                 'h':int(o.bbox.maxy-o.bbox.miny),
                 'cls':int(o.category.id), 'conf':float(o.score.value)}
                for o in res.object_prediction_list], 'sahi'
    r = yolo_model.predict(img_bgr,
                           imgsz=cfg['sahi_slice'],
                           conf=cfg['conf_threshold'],
                           iou=cfg['nms_iou'],
                           verbose=False)
    return [{'x':int(x1),'y':int(y1),'w':int(x2-x1),'h':int(y2-y1),
             'cls':int(b.cls),'conf':float(b.conf)}
            for res in r for b in res.boxes
            for x1,y1,x2,y2 in [b.xyxy[0].tolist()]], 'direct'


def run_yolo_folder(camera_dir, results_dir, cfg):
    camera_dir = Path(camera_dir); results_dir = Path(results_dir)
    crop_dir   = results_dir / 'yolo_crops'
    crop_dir.mkdir(parents=True, exist_ok=True)
    csv_path   = results_dir / 'yolo_results.csv'

    images = sorted(
        list(camera_dir.glob('*.JPG')) + list(camera_dir.glob('*.jpg')))
    if not images:
        print(f'  [SKIP] No images in {camera_dir.name}'); return 0

    cam    = camera_dir.name
    total  = len(images)
    method = 'sahi' if sahi_model else 'direct'
    print(f'  Images  : {total}')
    print(f'  Method  : {method}')
    print(f'  Processing...')

    n_det = 0; prog = cfg.get('progress_every', 50)
    t0 = time.time()

    with open(csv_path, 'w', newline='') as fh:
        writer = csv.DictWriter(fh, fieldnames=YOLO_CSV)
        writer.writeheader()

        for idx, path in enumerate(images):
            img = cv2.imread(str(path))
            if img is None: continue
            sh = cfg.get('strip_height', 0)
            if sh > 0 and img.shape[0] > sh:
                img = img[:img.shape[0]-sh, :]
            try:
                dets, det_method = detect_yolo(img, cfg)
            except Exception as e:
                print(f'  ⚠ {path.name}: {e}'); continue

            Hi, Wi = img.shape[:2]
            for i, d in enumerate(dets):
                cls_name = (YOLO_CLASSES[d['cls']]
                            if d['cls'] < len(YOLO_CLASSES) else str(d['cls']))
                crop_fn = ''
                if cfg.get('save_crops', True):
                    pad = cfg.get('crop_pad_px', 20)
                    x1 = max(0, d['x']-pad);      y1 = max(0, d['y']-pad)
                    x2 = min(Wi, d['x']+d['w']+pad); y2 = min(Hi, d['y']+d['h']+pad)
                    crop_fn = f'{cam}__{path.stem}_yolo{i:02d}_{cls_name}.jpg'
                    cv2.imwrite(str(crop_dir/crop_fn), img[y1:y2, x1:x2])
                    n_det += 1
                writer.writerow({
                    'camera_folder': cam, 'image_name': path.name,
                    'crop_filename': crop_fn,
                    'bbox_x': d['x'], 'bbox_y': d['y'],
                    'bbox_w': d['w'], 'bbox_h': d['h'],
                    'class_name': cls_name,
                    'confidence': round(d['conf'], 4),
                    'method': det_method,
                })

            if prog and (idx+1) % prog == 0:
                el = time.time()-t0; fps = (idx+1)/el if el else 0
                eta = (total-idx-1)/fps if fps else 0
                print(f'  [{idx+1:>5}/{total}] {fps:.1f} fps  '
                      f'ETA {eta:.0f}s  crops={n_det}', flush=True)

    el = time.time()-t0; fps = total/el if el else 0
    print(f'  ─────────────────────────────────────')
    print(f'  Done : {total} frames in {el:.1f}s ({fps:.1f} fps)')
    print(f'  Crops: {n_det} saved')
    return n_det


def get_leaf_dirs(root):
    root = Path(root)
    if (any(root.glob('*.JPG')) or any(root.glob('*.jpg'))) and \
       not any(p.is_dir() for p in root.iterdir()): return [root]
    return [d for d in sorted(root.rglob('*'))
            if d.is_dir() and not any(x.is_dir() for x in d.iterdir())
            and (any(d.glob('*.JPG')) or any(d.glob('*.jpg')))]

print('✓ Detection functions loaded.')


##### Cell 5 — Find camera folders
Scans `IMAGE_ROOT` for all camera subfolders and prints what will be processed.

In [ ]:
assert IMAGE_ROOT.exists(), f'IMAGE_ROOT not found: {IMAGE_ROOT}'
camera_dirs = get_leaf_dirs(IMAGE_ROOT)
total_images = 0
print(f'Found {len(camera_dirs)} camera folder(s):')
for d in camera_dirs:
    n = len(list(d.glob('*.JPG'))) + len(list(d.glob('*.jpg')))
    total_images += n
    print(f'  {d.name:<55} {n:>5} images')
print(f'  {"─"*62}')
print(f'  Total: {total_images} images')
print(f'\nResults → {RUN_DIR}')


##### Cell 6 — Run  ← main processing cell
Processes every camera folder and writes `yolo_results.csv` plus crop patches.

In [ ]:
import traceback

# Save run config
(RUN_DIR/'run_config.json').write_text(
    json.dumps({**YOLO_CONFIG,
                'run_name': RUN_NAME,
                'run_type': 'yolo',
                'weights':  str(YOLO_WEIGHTS),
                'classes':  YOLO_CLASSES}, indent=2))
print(f'✓ run_config.json saved')

stats = {'ok':0, 'fail':0, 'crops':0}
t_start = time.time()

for ci, camera_dir in enumerate(camera_dirs):
    out = RUN_DIR / camera_dir.relative_to(IMAGE_ROOT)
    out.mkdir(exist_ok=True)
    rel = camera_dir.relative_to(IMAGE_ROOT)
    print(f'\n[{ci+1}/{len(camera_dirs)}] {rel}')
    print(f'  ─────────────────────────────────────')
    try:
        n = run_yolo_folder(camera_dir, out, YOLO_CONFIG)
        stats['ok'] += 1; stats['crops'] += n
    except KeyboardInterrupt:
        print('\n⚠ Interrupted.'); raise
    except Exception as e:
        stats['fail'] += 1
        print(f'  ✗ ERROR: {e}'); traceback.print_exc()

total_time = time.time() - t_start
print(f'\n{"═"*55}')
print(f'RUN COMPLETE: {RUN_NAME}')
print(f'  Cameras : {stats["ok"]} ok  {stats["fail"]} failed')
print(f'  Crops   : {stats["crops"]} total')
print(f'  Time    : {total_time:.1f}s ({total_time/60:.1f} min)')
print(f'  Results : {RUN_DIR}')
print(f'  Config  : {RUN_DIR}/run_config.json')

# ── Auto-save results to Drive (Colab only) ──────────────────────
if IN_COLAB:
    import shutil
    drive_results = Path('/content/drive/MyDrive/pollinator-colab/Insects_images/yolo_results')
    drive_results.mkdir(parents=True, exist_ok=True)
    drive_run_dir = drive_results / RUN_NAME
    if drive_run_dir.exists():
        shutil.rmtree(str(drive_run_dir))
    print(f'\nSaving results to Drive...')
    shutil.copytree(str(RUN_DIR), str(drive_run_dir))
    print(f'✓ Results saved to Drive: {drive_run_dir}')


##### Cell 7 — Summary
Shows total detections and per-class breakdown after the run completes.

In [ ]:
all_csvs = list(RUN_DIR.rglob('yolo_results.csv'))
class_counts = {}; total = 0
for f in all_csvs:
    with open(f, newline='') as fh:
        for row in csv.DictReader(fh):
            total += 1
            c = row.get('class_name','')
            if c: class_counts[c] = class_counts.get(c,0) + 1
print(f'Run         : {RUN_NAME}')
print(f'SAHI        : {YOLO_CONFIG["use_sahi"]}  '
      f'conf={YOLO_CONFIG["sahi_conf"]}')
print(f'Total dets  : {total}')
if class_counts:
    print('Breakdown:')
    for c,n in sorted(class_counts.items(), key=lambda x:-x[1]):
        print(f'  {c:15}: {n}')
